# 📷 Img-Txt-Extractor — Colab Runner

Clones the repo, installs dependencies, starts the FastAPI backend, and exposes it over HTTPS via ngrok.

**Run all cells top-to-bottom (Runtime → Run all).**

In [ ]:
# ── Cell 1: Clone repo ─────────────────────────────────────────────────────
import os, subprocess, sys

REPO_URL  = "https://github.com/black1000u-blip/Img-Txt-Extractor.git"
REPO_DIR  = "Img-Txt-Extractor"

if os.path.isdir(REPO_DIR):
    print(f"[INFO] '{REPO_DIR}' already exists — pulling latest changes...")
    result = subprocess.run(["git", "-C", REPO_DIR, "pull"], capture_output=True, text=True)
    print(result.stdout or "(no output)")
    if result.stderr:
        print("[STDERR]", result.stderr)
else:
    print(f"[INFO] Cloning {REPO_URL} ...")
    result = subprocess.run(["git", "clone", REPO_URL], capture_output=True, text=True)
    print(result.stdout or "(no output)")
    if result.returncode != 0:
        print("[ERROR] git clone failed:")
        print(result.stderr)
        raise RuntimeError("git clone failed — see error above")
    print("[OK] Clone complete.")

os.chdir(REPO_DIR)
print(f"[INFO] Working directory: {os.getcwd()}")
print("[INFO] Repo contents:", os.listdir("."))

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────
import subprocess, sys

print("[INFO] Installing requirements (this may take 2–5 minutes on first run)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "backend/requirements.txt"],
    capture_output=False,   # stream output directly to Colab cell
    text=True
)
if result.returncode != 0:
    raise RuntimeError("pip install failed — see output above")
print("\n[OK] All packages installed.")

In [ ]:
# ── Cell 3: Start FastAPI backend (errors visible in this cell's output) ───
import threading, time, queue, sys, traceback

log_q = queue.Queue()
_server_ready = threading.Event()
_server_error: list = []

class QueueStream:
    """Pipe uvicorn stdout/stderr into the notebook cell."""
    def write(self, msg):
        if msg.strip():
            log_q.put(msg)
    def flush(self): pass

def run_server():
    try:
        import uvicorn
        # log_level='debug' → all request/error details visible
        uvicorn.run(
            "backend.app:app",
            host="0.0.0.0",
            port=3001,
            log_level="debug",    # ← change to 'info' for quieter output
            access_log=True,
        )
    except Exception:
        tb = traceback.format_exc()
        _server_error.append(tb)
        log_q.put("[SERVER ERROR]\n" + tb)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# Stream logs for 6 seconds so startup messages appear immediately
deadline = time.time() + 6
while time.time() < deadline:
    try:
        line = log_q.get(timeout=0.3)
        print(line, end="", flush=True)
    except queue.Empty:
        pass

if _server_error:
    raise RuntimeError("Server failed to start — see error above.")

print("\n[OK] Backend thread running on port 3001.")

# Background log printer (keeps errors visible after the deadline)
def _log_printer():
    while True:
        try:
            print(log_q.get(timeout=1), end="", flush=True)
        except queue.Empty:
            pass

threading.Thread(target=_log_printer, daemon=True).start()

In [ ]:
# ── Cell 4: Health-check — confirm server is actually responding ───────────
import urllib.request, urllib.error, time

def wait_for_server(url, retries=15, delay=2):
    for attempt in range(1, retries + 1):
        try:
            urllib.request.urlopen(url, timeout=3)
            return True
        except urllib.error.HTTPError as e:
            # 4xx/5xx from FastAPI still means the server is up
            return True
        except Exception as e:
            print(f"  [attempt {attempt}/{retries}] Not ready yet: {e}")
            time.sleep(delay)
    return False

print("[INFO] Waiting for server to be reachable at http://localhost:3001 ...")
ready = wait_for_server("http://localhost:3001")
if not ready:
    raise RuntimeError("Server did not respond after retries — check Cell 3 output for errors.")
print("[OK] Server is responding.")

In [ ]:
# ── Cell 5: Start ngrok tunnel ─────────────────────────────────────────────
from pyngrok import ngrok, conf
import traceback

NGROK_TOKEN = "3AbFUwuUiWdVY3KG97gvOnqiR18_7S56jZJzefkv21hPXGqar"

try:
    ngrok.set_auth_token(NGROK_TOKEN)
    tunnel = ngrok.connect(3001, "http")
    public_url = tunnel.public_url

    # Force HTTPS (ngrok always provides both; prefer https)
    if public_url.startswith("http://"):
        public_url = "https://" + public_url[len("http://"):]

    print("\n" + "═" * 60)
    print("  ✅  App is LIVE at:")
    print(f"      {public_url}")
    print("  Open this URL on your phone (camera requires HTTPS).")
    print("═" * 60 + "\n")

except Exception:
    print("[ERROR] ngrok tunnel failed:")
    traceback.print_exc()
    raise

In [ ]:
# ── Cell 6 (optional): Keep-alive — stream server logs until you stop ──────
# Run this cell to watch live server logs.
# Interrupt the kernel (■ Stop button) to stop it.
import time

print("[Live logs — press ■ Stop to exit]\n")
try:
    while True:
        try:
            line = log_q.get(timeout=1)
            print(line, end="", flush=True)
        except Exception:
            time.sleep(0.2)
except KeyboardInterrupt:
    print("\n[INFO] Log stream stopped.")